# <font color="brown">Recommendation Systems: Collaborative Filtering, Content-Based Filtering, and Hybrid Approaches</font>

## <font color="brown">Problem Statement</font>

### <font color="blue">Context</font>

A streaming service's movie catalog is far too large for any one person to browse. The whole
point of a recommendation system is to shortcut that browsing: look at what a user (and other
users) already liked, and surface a short, personalized list of what they'd probably like next.

There is more than one way to do this. This notebook builds two genuinely different approaches
from scratch, on the same small movie dataset, so their results can be compared directly:

- **Collaborative filtering**: recommend based on what *other, similar users* liked.
- **Content-based filtering**: recommend based on the *movie's own features* and this one user's
  own history.

It also builds a **hybrid** approach that blends the two, and demonstrates the **cold-start
problem**, the specific situation where each approach, on its own, quietly fails.

### <font color="blue">Objective</font>

- Build a small, synthetic movie-ratings dataset where the "right answer" is knowable, so every
  recommendation can be sanity-checked against it.
- Implement collaborative filtering and content-based filtering, from simple, from-scratch code,
  and compare what each one recommends for the same user.
- Show exactly where each approach breaks down (a brand-new user, a brand-new movie), and how a
  hybrid approach can help.

### <font color="blue">Data Dictionary</font>

All data here is synthetic, generated with a fixed random seed so every result in this notebook is
reproducible.

| Table | Column | What it means |
| --- | --- | --- |
| Movies | `Movie_ID` | A unique ID for each movie |
| Movies | `Title` | The movie's name |
| Movies | `Genre` | The movie's single genre (Action, Comedy, Drama, Romance, or Sci-Fi) |
| Ratings | rows = users, columns = movies | Each cell is a 1-5 star rating, or blank if that user never rated that movie |

## <font color="brown">Importing Necessary Libraries</font>

- `numpy` / `pandas`: generating and reshaping the data.
- `sklearn.metrics.pairwise.cosine_similarity`: the one similarity measurement this whole notebook
  is built on, explained in detail below.

In [1]:
import numpy as np
import pandas as pd

from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

## <font color="brown">Step 1: Reading and Reshaping Three Separate Files</font>

Real recommendation data almost never arrives as one ready-made matrix. It comes as separate
tables, typically a movie catalog, a users table, and a log of individual ratings, that have to be
joined together first. This notebook mirrors that: three CSV files, generated synthetically with a
fixed random seed (so every result below is reproducible), are read in and reshaped here exactly
the way a real system would have to.

- **`movies.csv`**: 20 movies, 4 in each of 5 genres (Action, Comedy, Drama, Romance, Sci-Fi).
- **`users.csv`**: 300 users. Each one also has a hidden `True_Favorite_Genre`, the genre they
  were secretly built to prefer, this column exists **only** because the data is synthetic and
  the answer needs to be checkable; a real system would never have this column.
- **`ratings.csv`**: one row per *actual* rating (not one row per user), the natural, sparse,
  "long format" a ratings log is stored in, each user only rated a handful of the 20 movies, not
  the whole catalog, matching how sparse real rating data actually is.

### <font color="blue">Reading the Three Files</font>

In [2]:
movies_df = pd.read_csv("movies.csv")
users_df = pd.read_csv("users.csv")
ratings_long_df = pd.read_csv("ratings.csv")

print("movies:", movies_df.shape)
print("users:", users_df.shape)
print("ratings (one row per actual rating):", ratings_long_df.shape)

movies: (20, 3)
users: (300, 2)
ratings (one row per actual rating): (2865, 3)


In [3]:
movies_df.head()

,Movie_ID,Title,Genre
0,0,Action Movie 1,Action
1,1,Action Movie 2,Action
2,2,Action Movie 3,Action
3,3,Action Movie 4,Action
4,4,Comedy Movie 1,Comedy


In [4]:
users_df.head()

,User_ID,True_Favorite_Genre
0,User_0,Action
1,User_1,Romance
2,User_2,Drama
3,User_3,Sci-Fi
4,User_4,Comedy


In [5]:
ratings_long_df.head()

,User_ID,Movie_ID,Rating
0,User_0,0,4
1,User_0,17,1
2,User_0,3,5
3,User_0,12,1
4,User_0,16,1


### <font color="blue">Reshaping the Ratings Log Into a User-Item Matrix</font>

`ratings.csv` has one row per rating, `User_ID`, `Movie_ID`, `Rating`. Collaborative filtering
needs the other shape instead: one row per **user**, one column per **movie**, so any two users'
full rating histories can be compared directly. Getting from one shape to the other takes two
steps: attach each movie's title (a simple merge/join), then pivot the long table into that wide
matrix.

In [6]:
# attach Title to every rating row, by matching Movie_ID against the movie catalog
ratings_with_titles = ratings_long_df.merge(movies_df[["Movie_ID", "Title"]], on="Movie_ID")

# reshape: one row per user, one column per movie title, cell = that user's rating
ratings_df = ratings_with_titles.pivot_table(index="User_ID", columns="Title", values="Rating")
ratings_df = ratings_df.reindex(users_df["User_ID"])   # keep every user, even ones with a fully blank row

ratings_df.iloc[:6, :6]

Title,Action Movie 1,Action Movie 2,Action Movie 3,Action Movie 4,Comedy Movie 1,Comedy Movie 2
User_ID,,,,,,
User_0,4.0,NaN,NaN,5.0,1.0,NaN
User_1,1.0,1.0,NaN,1.0,NaN,NaN
User_2,NaN,1.0,NaN,NaN,NaN,1.0
User_3,2.0,1.0,2.0,NaN,NaN,1.0
User_4,1.0,NaN,NaN,NaN,5.0,5.0
User_5,1.0,NaN,2.0,1.0,NaN,1.0


This is the **user-item ratings matrix**, the starting point for collaborative filtering. Most
cells are blank (`NaN`): a real ratings matrix is almost always mostly empty, since no user rates
more than a tiny fraction of a large catalog. `NaN` here means "never rated this movie," not "rated
it 0."

In [7]:
percent_filled = (~ratings_df.isna()).values.mean() * 100
print(f"Only {percent_filled:.1f}% of the ratings matrix is actually filled in")

Only 47.8% of the ratings matrix is actually filled in


## <font color="brown">Part 1: Collaborative Filtering</font>

**Core idea**: recommend based on what *other, similar users* liked, without ever looking at what
a movie is actually about. If a group of users has consistently agreed with this one user's taste
in the past, their other favorites are a good bet for this user too.

### <font color="blue">The Math: Measuring How Similar Two Users Are</font>

Every user's row in the ratings matrix is really just a list of numbers (their ratings, with
blanks treated as 0 for this comparison). **Cosine similarity** measures how similar two such
lists are, by treating each one as a direction, an arrow, and asking how closely those two arrows
point in the same direction. It gives a score from -1 (opposite directions) to 1 (exactly the same
direction); with ratings, which are never negative, the score in practice lands between 0 and 1.

The formula: for two rating vectors A and B,

`cosine_similarity(A, B) = (A · B) / (‖A‖ × ‖B‖)`

- `A · B` is the dot product: multiply matching elements together, then add them all up.
- `‖A‖` and `‖B‖` are each vector's length (magnitude).

Dividing by both lengths is what makes this a measure of *direction*, not overall size, a user who
rates everything a 5 and a user who rates everything a 4 can still come out as very similar, even
though their raw numbers differ.

In [8]:
# a tiny hand-checkable example: two users, ratings for just 3 shared movies
user_a = np.array([5, 4, 0])
user_b = np.array([4, 5, 1])

dot_product = np.sum(user_a * user_b)          # the A . B part of the formula
length_a = np.sqrt(np.sum(user_a ** 2))         # ||A||
length_b = np.sqrt(np.sum(user_b ** 2))         # ||B||

similarity_by_hand = dot_product / (length_a * length_b)
similarity_by_sklearn = cosine_similarity([user_a], [user_b])[0, 0]   # same formula, one function call

print("By hand:   ", round(similarity_by_hand, 4))
print("By sklearn:", round(similarity_by_sklearn, 4))

By hand:    0.9639
By sklearn: 0.9639


**Both give 0.9639.** These two users agree closely (a high score, close to 1), which makes sense:
both rated the first two movies highly and the third one low. Now the exact same calculation
(via `cosine_similarity`, not by hand from here on) is applied to every real user in the dataset.

### <font color="blue">Finding This User's Nearest Neighbors</font>

Now the same calculation, applied for real: compare one target user's full ratings row against
every other user's row, and keep the 15 most similar users, their "neighbors".

In [9]:
target_user = "User_0"

ratings_filled = ratings_df.fillna(0)   # cosine similarity needs real numbers, not blanks
target_row = ratings_filled.loc[[target_user]]   # double brackets keep this a table, not a single row

# compare this one user's row against every row (including their own) in one call
similarity_scores = cosine_similarity(target_row, ratings_filled)[0]
similarity_series = pd.Series(similarity_scores, index=ratings_df.index)
similarity_series = similarity_series.drop(index=target_user)   # a user is always identical to themself, drop that

K = 15
nearest_neighbors = similarity_series.sort_values(ascending=False).head(K)
nearest_neighbors.head(8)

User_ID
User_295    0.881662
User_273    0.858119
User_119    0.843077
User_163    0.827752
User_174    0.776320
User_31     0.776300
User_264    0.741249
User_183    0.739600
dtype: float64

In [10]:
# this check only works because True_Favorite_Genre exists, which only exists because this
# data is synthetic, a real dataset would have no such ground truth to check against
favorite_genre_lookup = users_df.set_index("User_ID")["True_Favorite_Genre"]

target_favorite_genre = favorite_genre_lookup[target_user]
neighbor_favorite_genres = favorite_genre_lookup[nearest_neighbors.index].tolist()

print("Target user's hidden favorite genre:", target_favorite_genre)
print("Neighbors' hidden favorite genres:  ", neighbor_favorite_genres)

match_rate = np.mean([g == target_favorite_genre for g in neighbor_favorite_genres])
print(f"\nShare of neighbors who secretly share the target user's favorite genre: {match_rate:.0%}")

Target user's hidden favorite genre: Action
Neighbors' hidden favorite genres:   ['Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Comedy', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action', 'Action']

Share of neighbors who secretly share the target user's favorite genre: 93%


**93% of this user's 15 nearest neighbors secretly share their favorite genre (Action)**, found
using nothing but rating patterns, cosine similarity never saw the word "Action" anywhere. This is
the check that matters: since this data's hidden ground truth is known, this confirms
collaborative filtering is finding genuinely similar taste, not a coincidence.

### <font color="blue">Predicting Ratings From the Neighbors, and Recommending</font>

The idea from here is simple: look at what the neighbors rated highly, and recommend the movies
they liked that the target user hasn't seen yet. Averaging the neighbors' ratings for each movie
gives a predicted rating; one extra safeguard is worth adding, only trust that average if at least
3 of the 15 neighbors actually rated that movie, otherwise it is just one or two people's opinion,
not a real pattern.

In [11]:
neighbors_ratings = ratings_df.loc[nearest_neighbors.index]   # just these 15 users' rows

# for each movie: how many of the 15 neighbors rated it, and what did they rate it on average
how_many_neighbors_rated = neighbors_ratings.notna().sum(axis=0)
predicted_rating = neighbors_ratings.mean(axis=0, skipna=True)   # skipna ignores the blanks

# only trust a predicted rating if at least 3 neighbors actually contributed to it
enough_neighbors_rated = how_many_neighbors_rated >= 3
predicted_rating = predicted_rating[enough_neighbors_rated]

already_seen = ratings_df.loc[target_user].dropna().index
predicted_rating = predicted_rating.drop(index=already_seen, errors="ignore")   # never recommend what they've already rated

cf_recommendations = predicted_rating.sort_values(ascending=False).head(5)
cf_recommendations

Title
Action Movie 3     4.714286
Action Movie 2     3.800000
Romance Movie 3    1.500000
Sci-Fi Movie 3     1.444444
Sci-Fi Movie 4     1.400000
dtype: float64

**The top two recommendations, `Action Movie 3` (predicted 4.71) and `Action Movie 2` (predicted
3.80), are both Action movies this user hasn't seen yet**, exactly matching their hidden true
taste. The next three (Romance, Sci-Fi) sit far lower, around 1.4-1.5, correctly reflecting that
this user is not really an Action fan's Romance or Sci-Fi tastes, weak signal from a handful of
neighbors who happened to rate those movies too.

## <font color="brown">Part 2: Content-Based Filtering</font>

**Core idea**: recommend based on the *movie's own features*, and what this one user has liked
before. No other users are involved at all here, this whole approach works with a single user's
own history and a description of every movie.

### <font color="blue">Describing Every Movie With Simple Features</font>

Genre is the only feature available in this dataset, so it becomes a simple **one-hot encoded**
vector for every movie: a checklist of 5 yes/no columns (one per genre), with a `1` in the one
column that matches that movie's genre and `0` everywhere else.

In [12]:
genre_features = pd.get_dummies(movies_df.set_index("Title")["Genre"]).astype(float)   # one column per genre
genre_features.head(6)

,Action,Comedy,Drama,Romance,Sci-Fi
Title,,,,,
Action Movie 1,1.0,0.0,0.0,0.0,0.0
Action Movie 2,1.0,0.0,0.0,0.0,0.0
Action Movie 3,1.0,0.0,0.0,0.0,0.0
Action Movie 4,1.0,0.0,0.0,0.0,0.0
Comedy Movie 1,0.0,1.0,0.0,0.0,0.0
Comedy Movie 2,0.0,1.0,0.0,0.0,0.0


### <font color="blue">Building a Profile of What This User Likes</font>

Take every movie this user rated highly (a 4 or a 5), and average their feature vectors together.
The result is a simple numeric summary of this one user's taste, built entirely from their own
history.

In [13]:
user_ratings = ratings_df.loc[target_user].dropna()
liked_movies = user_ratings[user_ratings >= 4].index   # only the movies they actually loved

print("Movies this user rated 4 or 5 stars:", list(liked_movies))

# average the genre vectors of just those liked movies, one number per genre = the user's profile
user_profile = genre_features.loc[liked_movies].mean(axis=0)
user_profile

Movies this user rated 4 or 5 stars: ['Action Movie 1', 'Action Movie 4']


Action     1.0
Comedy     0.0
Drama      0.0
Romance    0.0
Sci-Fi     0.0
dtype: float64

Both of this user's highly-rated movies happen to be Action movies, so their profile comes out as
**pure Action (1.0), and zero everywhere else**. That is a direct, honest reflection of their
actual rating history, no other users, no assumptions, just this one person's own choices.

### <font color="blue">Recommending the Most Similar Movies</font>

The same cosine similarity from Part 1, applied differently here: compare this one user's profile
against every movie's feature vector, and recommend whichever movies are most similar, excluding
anything already seen.

In [14]:
content_similarity = cosine_similarity(
    user_profile.values.reshape(1, -1), genre_features.values   # reshape: sklearn wants a 2D table, even for one row
)[0]

content_scores = pd.Series(content_similarity, index=genre_features.index)
content_scores = content_scores.drop(index=already_seen, errors="ignore")

cb_recommendations = content_scores.sort_values(ascending=False).head(5)
cb_recommendations

Title
Action Movie 2    1.0
Action Movie 3    1.0
Comedy Movie 2    0.0
Comedy Movie 4    0.0
Drama Movie 1     0.0
dtype: float64

**`Action Movie 2` and `Action Movie 3` both score a perfect 1.0**, since they are pure Action, an
exact match to this user's pure-Action profile. Every non-Action movie scores exactly 0.0, since a
one-hot genre vector for a different genre points in a completely different direction, cosine
similarity of 0 means "unrelated," not "disliked."

## <font color="brown">Part 3: Comparing the Two Approaches, Side by Side</font>

Two completely different approaches, one using 300 other people's behavior and never looking at
what a movie is about, the other using only this one user's own history and the movie's genre.
Do they actually agree?

In [15]:
comparison = pd.DataFrame({
    "Collaborative Filtering": cf_recommendations,
    "Content-Based Filtering": cb_recommendations,
}).round(3)
comparison

,Collaborative Filtering,Content-Based Filtering
Title,,
Action Movie 2,3.800,1.0
Action Movie 3,4.714,1.0
Comedy Movie 2,NaN,0.0
Comedy Movie 4,NaN,0.0
Drama Movie 1,NaN,0.0
Romance Movie 3,1.500,NaN
Sci-Fi Movie 3,1.444,NaN
Sci-Fi Movie 4,1.400,NaN


**Yes.** Both approaches independently put the exact same two movies, `Action Movie 3` and
`Action Movie 2`, at the top. Collaborative filtering additionally distinguishes between them
(4.71 vs. 3.80, since real neighbors rated one slightly higher on average), while content-based
filtering treats them as tied (both a perfect genre match). Two approaches, built from completely
different information, one from 300 other people's behavior, the other from one person's own
history and a movie's genre, landing on the same answer is real evidence this recommendation is
correct, not a fluke of either method.

## <font color="brown">Part 4: The Cold-Start Problem</font>

Both approaches above were built using the user's own ratings, collaborative filtering to find
similar people, content-based filtering to build a profile. What happens when that history simply
does not exist yet?

### <font color="blue">A Brand-New User, With Zero Ratings</font>

Simulate the most common real case: someone who just signed up, with an empty ratings row.

In [16]:
new_user_ratings = pd.Series(0, index=ratings_df.columns)   # every movie unrated = an all-zero row

new_user_similarity = cosine_similarity(
    new_user_ratings.values.reshape(1, -1), ratings_filled.values
)[0]

print("Highest similarity to any existing user:", new_user_similarity.max())

Highest similarity to any existing user: 0.0


**Exactly 0.0, to every single existing user.** With an all-zero ratings row, there is nothing to
compare, the dot product with every other user is 0, so cosine similarity is 0 across the board.
Collaborative filtering has no way to find "similar" users for someone with no history at all, this
is not a bug to fix, it is a structural limitation of the approach itself.

Content-based filtering has a way around this, though: it does not need *ratings* history, only
*some* signal about taste. Asking a new user one quick preference question ("pick a genre you
like") is often enough to build a first profile immediately, no ratings required at all.

In [17]:
stated_preference = "Sci-Fi"

# build a profile directly from one stated preference, no ratings involved at all
quick_profile = pd.Series(0.0, index=genre_features.columns)
quick_profile[stated_preference] = 1.0

quick_content_similarity = cosine_similarity(
    quick_profile.values.reshape(1, -1), genre_features.values
)[0]

quick_recommendations = pd.Series(quick_content_similarity, index=genre_features.index)
quick_recommendations.sort_values(ascending=False).head(5)

Title
Sci-Fi Movie 4    1.0
Sci-Fi Movie 3    1.0
Sci-Fi Movie 2    1.0
Sci-Fi Movie 1    1.0
Action Movie 4    0.0
dtype: float64

**All 4 Sci-Fi movies immediately score a perfect 1.0**, from a single stated preference and zero
ratings. This is content-based filtering's real advantage at cold start: it only needs *some*
signal about taste, not a rating history built up over time.

### <font color="blue">A Brand-New Movie, Never Rated by Anyone</font>

The mirror-image problem: a movie just added to the catalog, with zero ratings from anyone at all.

In [18]:
new_movie_title = "Action Movie 5"
new_movie_genre = "Action"

ratings_df[new_movie_title] = np.nan  # not one single rating exists for this movie yet

print("How many ratings does", new_movie_title, "have?", ratings_df[new_movie_title].notna().sum())
# reindex adds this new, all-blank column to the old neighbors_ratings table so it can be averaged the same way
print("Average rating from this movie's 15 nearest-neighbor group:", neighbors_ratings.reindex(columns=[new_movie_title]).mean(axis=0).iloc[0])

How many ratings does Action Movie 5 have? 0
Average rating from this movie's 15 nearest-neighbor group: nan


**Zero ratings, and the neighbor-group average comes out as `NaN`, undefined.** There is no data
to average. No matter how good this movie might actually be, or how good a match it is for any
particular user, collaborative filtering cannot recommend it to *anyone* until at least a few
people rate it first, a structural blind spot for new items, not just new users.

In [19]:
# add this new movie's genre row to the feature table, purely by its genre, no ratings needed
extended_genre_features = pd.concat([
    genre_features,
    pd.DataFrame([[1.0 if g == new_movie_genre else 0.0 for g in genre_features.columns]],
                 columns=genre_features.columns, index=[new_movie_title]),
])

new_movie_score = cosine_similarity(
    user_profile.values.reshape(1, -1),
    extended_genre_features.loc[[new_movie_title]].values,
)[0, 0]

print(f"Content-based similarity between this user's profile and {new_movie_title}: {new_movie_score:.3f}")

Content-based similarity between this user's profile and Action Movie 5: 1.000


**A perfect 1.000, despite zero ratings.** Content-based filtering only needed this new movie's
genre, not a single rating from anyone, to confidently recommend it to this Action-loving user.
This is the exact mirror image of the new-user case above: content-based filtering solves *both*
cold-start problems, because it never depended on rating history in the first place.

## <font color="brown">Part 5: Hybrid Recommendations</font>

A hybrid system simply blends the two scores together, so the weaknesses of one are covered by the
strength of the other. One detail matters first: collaborative filtering's scores are 1-5 star
predictions, while content-based filtering's scores are 0-1 similarity scores, two different
scales that cannot be added together directly. **Min-max scaling** first rescales both onto the
same 0-1 range, so a 50/50 blend actually means 50/50, not accidentally dominated by whichever
score happens to have bigger raw numbers.

In [20]:
def min_max_scale(scores):
    # squashes any set of numbers down to the 0-1 range, so CF and CB become directly comparable
    return (scores - scores.min()) / (scores.max() - scores.min())

# a movie might have a CF score but no CB score, or vice versa, so consider every movie either method scored
candidate_movies = predicted_rating.index.union(content_scores.index)

cf_for_blend = predicted_rating.reindex(candidate_movies)   # missing movies become NaN here
cb_for_blend = content_scores.reindex(candidate_movies)

# fill any gaps with that method's own lowest score, a neutral "this method has nothing to say" stand-in
cf_scaled = min_max_scale(cf_for_blend.fillna(cf_for_blend.min()))
cb_scaled = min_max_scale(cb_for_blend.fillna(cb_for_blend.min()))

hybrid_score = 0.5 * cf_scaled + 0.5 * cb_scaled   # a simple 50/50 blend of the two scaled scores

hybrid_recommendations = hybrid_score.sort_values(ascending=False).head(5)
hybrid_recommendations

Title
Action Movie 3     1.000000
Action Movie 2     0.873128
Romance Movie 3    0.053965
Sci-Fi Movie 3     0.046256
Sci-Fi Movie 4     0.040088
dtype: float64

**`Action Movie 3` and `Action Movie 2` remain clearly on top (1.000 and 0.873)**, consistent with
both individual approaches, then a sharp drop to everything else. The blend didn't change the
answer here, both methods already agreed, but on a case where they disagreed, this same blend
would let each approach's signal genuinely count, rather than trusting only one method's opinion.

## <font color="brown">Practical Insights and Recommendations</font>

- **Collaborative and content-based filtering, built completely independently, recommended the
  same movies for this user.** That agreement is real evidence the recommendations reflect this
  user's actual taste, not an artifact of either method.
- **Collaborative filtering cannot help a brand-new user or a brand-new item, structurally, not as
  a bug.** With no ratings, there is nothing to compare, similarity to every other user comes out
  as zero.
- **Content-based filtering can serve both cold-start cases immediately**, a new user needs only a
  stated preference (not a rating history), and a new item needs only its own features (not
  anyone's rating of it) to get a real recommendation score.
- **A hybrid approach costs little and buys real robustness**: blend the two (after scaling them
  onto the same range), lean on content-based filtering during cold start, and let collaborative
  filtering's community signal take over as real rating history builds up.
- **The general pattern matters more than movies specifically.** Swap "movie" for "product,"
  "genre" for "category," and "rating" for "purchase," and the exact same two approaches, and the
  exact same cold-start problem, apply unchanged.